# Análise exploratória da qualidade das marcas

Este notebook estima quanto da coluna `marca` possui informações válidas. A avaliação é feita em etapas, nesta ordem:

1. valores formados somente por números;
2. descrições que aparentam não ser marcas (unidades, embalagens, dosagens ou marcadores genéricos);
3. valores nulos ou em branco;
4. estimativa de marcas **válidas ou possivelmente válidas**.

> A classificação é heurística: o grupo final não representa marcas confirmadas, mas valores que não foram rejeitados pelas regras desta análise.

## 1. Leitura dos dados

In [117]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)

# Localiza a raiz do repositório mesmo quando o notebook é aberto a partir de outra pasta.
raiz = next(
    caminho
    for caminho in (Path.cwd(), *Path.cwd().parents)
    if (caminho / "tasks/api-compras").is_dir()
)
arquivo = raiz / "tasks/api-compras/docs/inputs/item_homologado_marcas.csv"
# Todas as colunas são lidas como texto para preservar marcas numéricas e zeros à esquerda.
dados = pd.read_csv(arquivo, dtype=str)

colunas = [
    "numero_controle_pncp",
    "marca",
    "descricao_detalhada_item",
    "sigla_unidade_fornecimento",
    "nome_unidade_fornecimento",
]
dados = dados[colunas].copy()

print(f"Arquivo: {arquivo.name}")
print(f"Total de linhas de itens: {len(dados):,}".replace(",", "."))
display(dados.head())

Arquivo: item_homologado_marcas.csv
Total de linhas de itens: 65.976


,numero_controle_pncp,marca,descricao_detalhada_item,sigla_unidade_fornecimento,nome_unidade_fornecimento
0,00000368000150-1-000039/2025,DFL,"MEPIVACAÍNA CLORIDRATO, APRESENTAÇÃO ASSOCIADA COM EPINEFRINA, DOSAGEM 2% +1:100.000",TBTE,TUBETE
1,00000368000150-1-000039/2025,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TBTE,TUBETE
2,00000368000150-1-000039/2025,DFL,"BENZOCAÍNA, CONCENTRAÇÃO 20%, USO GEL TÓPICO",POTE,POTE
3,00000368000150-1-000039/2025,IODONTOSUL,"BICARBONATO DE SÓDIO, APRESENTAÇÃO PÓ",SAC,SACHÊ
4,00000368000150-1-000046/2024,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TBTE,TUBETE


## 2. Preparação e regras da análise

A normalização remove acentos, uniformiza maiúsculas e espaços apenas para permitir comparações. A coluna original é preservada nos resultados.

As máscaras abaixo são independentes para facilitar a auditoria. No resumo final, cada linha recebe uma única classificação, usando a ordem apresentada no início do notebook.

In [118]:
# Mantém o valor original para exibição e cria uma versão limpa para as comparações.
marca_original = dados["marca"]
marca_texto = marca_original.fillna("").str.strip()
# A normalização não altera o dado de origem: ela apenas reduz diferenças de acentuação,
# caixa e espaçamento que poderiam esconder valores equivalentes.
marca_normalizada = (
    marca_texto.str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
    .str.upper()
    .str.replace(r"[^A-Z0-9%/.,:+-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
# Remove somente a pontuação das bordas para reconhecer casos como `.Ampola de vidro.`
# e `/CX C/60 FRS`, sem alterar o valor original nem a normalização-base.
marca_normalizada_sem_pontuacao_borda = marca_normalizada.str.strip(" .,/;:_-")

# Respostas administrativas e textos genéricos que não identificam um fabricante ou marca.
marcadores_genericos_originais = {
    "NAO SE APLICA", "NAO APLICAVEL", "SEM MARCA", "S MARCA",
    "MARCA NAO INFORMADA", "NAO INFORMADO", "NAO INFORMADA",
    "IGNORADO", "INDEFINIDO", "DIVERSOS", "DIVERSAS",
    "GENERICO", "GENERICA", "SIMILAR", "MANIPULADO", "MANIPULADA",
    "MEDICAMENTO MANIPULA", "MEDICAMENTO MANIPULADO", "PROPRIA", "MARCA PROPRIA",
    "NACIONAL", "IMPORTADO", "A DEFINIR", "CONFORME EDITAL", "CONFORME TR",
    "CONFORME TERMO DE REFERENCIA", "CONFORME DESCRICAO",
    "SEM INFORMACAO", "NA", "N A", "NC", "N C", "SN", "S N", "-1023509900", "A", "A 600",
    "A0959", "A2F", "A6279", "A?CIDO ZOLEDRO?NICO", "AA", "AAF DO BRASIL",
}

# Converte a coluna marca da codelist externa em uma lista normalizada. Manter o CSV como
# fonte da verdade facilita revisar, acrescentar ou remover termos sem editar centenas de
# literais dentro do notebook.
arquivo_codelist = raiz / "tasks/api-compras/docs/inputs/codelist_infos_erradas.csv"
marcas_codelist = pd.read_csv(
    arquivo_codelist, usecols=["marca"], dtype=str
)["marca"].dropna().str.strip()
marcadores_genericos_codelist = (
    marcas_codelist.str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("ascii")
    .str.upper()
    .str.replace(r"[^A-Z0-9%/.,:+-]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .loc[lambda valores: valores.ne("")]
    .drop_duplicates()
    .tolist()
)

# A codelist existente é incrementada com os  valores normalizados do CSV.
marcadores_genericos = marcadores_genericos_originais | set(marcadores_genericos_codelist)

# Vocabulário observado em valores que descrevem apresentação, recipiente ou unidade.
termos_unidade_embalagem = (
    r"UN|UND|UNID|UNIDADE|UNIDADES|CX|CAIXA|CAIXAS|CT|CARTELA|CARTELAS|"
    r"COMPRIMIDO|COMPRIMIDOS|COMP|CPR|CAPSULA|CAPSULAS|CAP|AMP|AMPOLA|"
    r"AMPOLAS|FR|FRASCO|FRASCOS|FA|BISNAGA|BOLSA|ENVELOPE|FLACONETE|"
    r"GALAO|KIT|LATA|PACOTE|PCT|POTE|SACHE|SERINGA|TBTE|TUBETE|TUBO|VIDRO"
)
# A versão ampliada é usada somente pelas novas regras, preservando o vocabulário histórico
# acima para que o comparativo antes × depois seja fiel à execução anterior.
termos_unidade_embalagem_ampliados = (
    rf"{termos_unidade_embalagem}|GALOES|KITS|LATAS|PACOTES|POTES|SACHES|"
    r"SERINGAS|TUBETES|TUBOS|VIDROS"
)
# fullmatch exige que o valor inteiro tenha formato de unidade/embalagem. Isso evita,
# por exemplo, rejeitar uma marca legítima que contenha uma dessas palavras no nome.
unidade_embalagem = marca_normalizada.str.fullmatch(
    rf"(?:{termos_unidade_embalagem})(?:[\s/-]+(?:{termos_unidade_embalagem}))*"
    rf"(?:[\s/+-]+(?:C|COM|CONTENDO|DE)?\s*\d.*)?",
    na=False,
)

# Reconhece valores compostos essencialmente por quantidade e unidade de concentração.
dosagem = marca_normalizada.str.fullmatch(
    r"\d+(?:[.,]\d+)?\s*(?:MG|G|MCG|UG|KG|ML|L|UI|U|MEQ|MMOL|%)"
    r"(?:\s*/\s*(?:ML|L|DOSE|COMPRIMIDO|CAPSULA))?(?:\s+.*)?",
    na=False,
)

# As máscaras permanecem separadas para que cada hipótese possa ser auditada isoladamente.
# Nulo (NaN) e texto em branco são problemas distintos, embora ambos não tenham conteúdo útil.
nula = marca_original.isna()
em_branco = ~nula & marca_texto.eq("")
somente_numerica = marca_texto.str.fullmatch(r"\d+(?:[.,]\d+)?", na=False)
marcador_generico_original = marca_normalizada.isin(marcadores_genericos_originais)
marcador_codelist = marca_normalizada.isin(marcadores_genericos_codelist)
marcador_generico = marca_normalizada.isin(marcadores_genericos)

# Guarda a regra anterior à melhoria de formatação. Esta máscara permite medir no próprio
# notebook quantas linhas foram reclassificadas pelas novas regras, sem depender de uma
# execução ou de um arquivo antigo.
descricao_nao_marca_antes = marcador_generico_original | unidade_embalagem | dosagem

# Casos de formatação sem conteúdo semântico: somente sinais ou textos iniciados por aspas.
# marca_texto já teve espaços externos removidos, portanto a âncora ^ aponta para o primeiro
# caractere efetivamente informado.
somente_pontuacao = marca_texto.str.fullmatch(r"[\W_]+", na=False)
inicia_com_aspas = marca_texto.str.match(r'''^["'“”‘’´`]''', na=False)
formato_invalido = somente_pontuacao | inicia_com_aspas

# O fullmatch anterior continua sendo a regra mais segura. Estas máscaras adicionais capturam
# descrições evidentemente iniciadas por embalagem ou apresentação quantitativa, ainda que o
# restante do campo contenha detalhes, registro sanitário ou texto livre.
embalagem_no_inicio = marca_normalizada.str.match(
    rf"^(?:{termos_unidade_embalagem_ampliados})\b", na=False
)
dosagem_composta_no_inicio = marca_normalizada.str.match(
    r"^\d+(?:[.,]\d+)?(?:\s*\+\s*\d+(?:[.,]\d+)?)+\s*"
    r"(?:MG|G|MCG|UG|KG|ML|L|UI|U|MEQ|MMOL|%)\b",
    na=False,
)
quantidade_embalagem_no_inicio = marca_normalizada.str.match(
    rf"^\d+\s*(?:{termos_unidade_embalagem_ampliados})\b", na=False
)
descricao_apresentacao = (
    embalagem_no_inicio | dosagem_composta_no_inicio | quantidade_embalagem_no_inicio
)

# Bloco de regras de formatação e apresentação que faz parte da segunda rodada consolidada.
descricao_nao_marca_formatacao_apresentacao = (
    descricao_nao_marca_antes | formato_invalido | descricao_apresentacao
)

# Por decisão metodológica, valores iniciados por GENÉRICO/GENÉRICA permanecem no conjunto
# possivelmente válido, mesmo quando trazem laboratório, registro ou apresentação.
generico_para_manter = marca_normalizada_sem_pontuacao_borda.str.match(
    r"^(?:GENERICO|GENERICA|MEDICAMENTO GENERICO)\b", na=False
)

# Captura expressões formadas apenas por duas ou mais parcelas numéricas.
expressao_numerica = marca_normalizada_sem_pontuacao_borda.str.fullmatch(
    r"\(?\s*\d+(?:[.,]\d+)?(?:\s*\+\s*\d+(?:[.,]\d+)?){1,}\s*\)?",
    na=False,
)

# Dosagens no início seguidas por qualquer detalhe de apresentação.
dosagem_no_inicio = marca_normalizada_sem_pontuacao_borda.str.match(
    r"^\d+(?:[.,]\d+)?\s*(?:MG|G|MCG|UG|KG|ML|L|UI|U|MEQ|MMOL|%)\b",
    na=False,
)

# Reaplica a regra de embalagem após retirar pontos, barras e outros sinais das bordas.
embalagem_com_pontuacao = marca_normalizada_sem_pontuacao_borda.str.match(
    rf"^(?:{termos_unidade_embalagem_ampliados})\b", na=False
)

# Registros sanitários e códigos estruturados. Os formatos são específicos para evitar que
# toda marca alfanumérica seja classificada como código.
registro_no_inicio = marca_normalizada_sem_pontuacao_borda.str.match(
    r"^(?:ANVISA|RMS|REG|REGISTRO)\b\s*[:.-]*\s*\d{6,}", na=False
)
codigo_estruturado = marca_normalizada_sem_pontuacao_borda.str.fullmatch(
    r"(?:\d{2}[A-Z]\d{4}\.\d{2}\.[A-Z]{2}|[A-Z]{2}\d{5}[A-Z]{2}|"
    r"[A-Z]{2,4}-\d{2,}(?:-\d+)*|[A-Z]\d{3,}(?:-\d+)+(?:MG|G|ML|L|KG)?|"
    r"[A-Z]{1,4}\d{1,4}-\d{2,}|\d{5,}[A-Z]{2,})",
    na=False,
)

regras_codigos_dosagens = (
    expressao_numerica | dosagem_no_inicio | embalagem_com_pontuacao |
    registro_no_inicio | codigo_estruturado
)
# Preserva o resultado da segunda rodada antes da codelist para medir seu impacto isolado.
descricao_nao_marca_sem_codelist = (
    descricao_nao_marca_formatacao_apresentacao | regras_codigos_dosagens
) & ~generico_para_manter
# A exceção dos genéricos vale também para os valores presentes na codelist.
descricao_nao_marca = (
    descricao_nao_marca_sem_codelist | marcador_codelist
) & ~generico_para_manter

dados["marca_normalizada"] = marca_normalizada
# Registra o motivo no DataFrame para facilitar resumos e inspeção dos exemplos.
dados["motivo_descricao_ruim"] = pd.NA
dados.loc[marcador_generico, "motivo_descricao_ruim"] = "marcador genérico"
dados.loc[unidade_embalagem, "motivo_descricao_ruim"] = "unidade/embalagem"
dados.loc[dosagem | dosagem_composta_no_inicio, "motivo_descricao_ruim"] = "dosagem/concentração"
dados.loc[
    embalagem_no_inicio | quantidade_embalagem_no_inicio,
    "motivo_descricao_ruim",
] = "unidade/embalagem"
dados.loc[formato_invalido, "motivo_descricao_ruim"] = "formatação inválida"
dados.loc[expressao_numerica, "motivo_descricao_ruim"] = "expressão numérica"
dados.loc[dosagem_no_inicio, "motivo_descricao_ruim"] = "dosagem/concentração"
dados.loc[embalagem_com_pontuacao, "motivo_descricao_ruim"] = "unidade/embalagem"
dados.loc[registro_no_inicio | codigo_estruturado, "motivo_descricao_ruim"] = "registro/código"
dados.loc[generico_para_manter, "motivo_descricao_ruim"] = pd.NA
dados

,numero_controle_pncp,marca,descricao_detalhada_item,sigla_unidade_fornecimento,nome_unidade_fornecimento,marca_normalizada,motivo_descricao_ruim
0,00000368000150-1-000039/2025,DFL,"MEPIVACAÍNA CLORIDRATO, APRESENTAÇÃO ASSOCIADA COM EPINEFRINA, DOSAGEM 2% +1:100.000",TBTE,TUBETE,DFL,<NA>
1,00000368000150-1-000039/2025,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TBTE,TUBETE,DFL,<NA>
2,00000368000150-1-000039/2025,DFL,"BENZOCAÍNA, CONCENTRAÇÃO 20%, USO GEL TÓPICO",POTE,POTE,DFL,<NA>
3,00000368000150-1-000039/2025,IODONTOSUL,"BICARBONATO DE SÓDIO, APRESENTAÇÃO PÓ",SAC,SACHÊ,IODONTOSUL,<NA>
4,00000368000150-1-000046/2024,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TBTE,TUBETE,DFL,<NA>
...,...,...,...,...,...,...,...
65971,96291141000180-1-008869/2024,AUROBINDO,Finasterida concentração: 5,CAPS,CÁPSULA,AUROBINDO,<NA>
65972,96291141000180-1-008869/2024,MEDLEY,"Travoprosta dosagem: 0,04, apresentação: solução oftálmica",FR,FRASCO,MEDLEY,<NA>
65973,96480850000103-1-000011/2025,Clarity Alcool liq,"ÁLCOOL ETÍLICO, TIPO HIDRATADO, TEOR ALCOÓLICO 70%_(70°GL), APRESENTAÇÃOLÍQUIDO",FR,FRASCO,CLARITY ALCOOL LIQ,<NA>
65974,96480850000103-1-000011/2025,Clarity Alcool gel,"ÁLCOOL ETÍLICO, TEOR ALCOÓLICO 70% V/V, COMPOSIÇÃO BÁSICA COM EMOLIENTE, FORMAFARMACÊUTICA GEL",GL,GALÃO,CLARITY ALCOOL GEL,<NA>


## 3. Existem valores somente numéricos?

São considerados somente numéricos valores compostos por dígitos, com separador decimal opcional. A tabela mostra todos os valores encontrados e sua frequência.

In [119]:
# A cópia permite explorar este subconjunto sem modificar o DataFrame completo.
numericos = dados.loc[somente_numerica].copy()
qtd_numericos = len(numericos)
# percentual de valores numéricos em relação ao total de linhas
pct_numericos = qtd_numericos / len(dados) * 100 if len(dados) else 0

print(f"Linhas somente numéricas: {qtd_numericos:,} ({pct_numericos:.2f}%)".replace(",", "."))

if numericos.empty:
    print("Nenhum valor somente numérico foi encontrado.")
else:
    # Agrupar por valor revela repetições.
    frequencia_numericos = (
        numericos.groupby("marca", dropna=False)
        .size().rename("quantidade").sort_values(ascending=False).to_frame()
    )
    display(frequencia_numericos)

Linhas somente numéricas: 798 (1.21%)


,quantidade
marca,
1,67
500.0000,12
2025,10
200.0000,9
50.0000,9
...,...
86000.0000,1
88192,1
7150.0000,1


## 4. Existem descrições que aparentam não ser marcas?

Aqui entram marcadores genéricos (`SEM MARCA`, `NÃO INFORMADO` etc.), unidades ou embalagens (`CAIXA`, `AMPOLA`, `FRASCO` etc.), dosagens/concentrações (`500 MG`, `10 MG/ML` etc.) e formatos evidentemente inválidos (somente pontuação ou textos iniciados por aspas).

O `fullmatch` permanece como regra principal. Regras ancoradas no início do texto complementam a captura quando o campo começa claramente por uma embalagem, quantidade ou dosagem e depois traz outros detalhes. Assim, uma marca que apenas contenha um desses termos no meio do nome não é rejeitada automaticamente.

Além das regex, a coluna `marca` de `inputs/codelist_infos_erradas.csv` é convertida em uma lista normalizada e incorporada a `marcadores_genericos`. O impacto dessa codelist é medido separadamente no comparativo.

In [120]:
# Exclui os somente numéricos deste recorte para não contabilizar a mesma linha duas vezes.
descricoes_ruins = dados.loc[descricao_nao_marca & ~somente_numerica].copy()
qtd_descricoes_ruins = len(descricoes_ruins)
pct_descricoes_ruins = qtd_descricoes_ruins / len(dados) * 100 if len(dados) else 0

print(
    f"Descrições que aparentam não ser marcas: {qtd_descricoes_ruins:,} "
    f"({pct_descricoes_ruins:.2f}%)".replace(",", ".")
)

if descricoes_ruins.empty:
    print("Nenhuma descrição incompatível com marca foi encontrada pelas regras atuais.")
else:
    # Primeiro resume por tipo de problema; depois detalha cada valor encontrado.
    resumo_descricoes = (
        descricoes_ruins.groupby("motivo_descricao_ruim")
        .size().rename("quantidade").sort_values(ascending=False).to_frame()
    )
    resumo_descricoes["percentual_do_total"] = (
        resumo_descricoes["quantidade"] / len(dados) * 100
    ).round(2)
    display(resumo_descricoes)

    frequencia_descricoes = (
        descricoes_ruins.groupby(["motivo_descricao_ruim", "marca_normalizada"])
        .size().rename("quantidade").reset_index()
        .sort_values(["motivo_descricao_ruim", "quantidade", "marca_normalizada"],
                     ascending=[True, False, True])
    )
    display(frequencia_descricoes)

Descrições que aparentam não ser marcas: 17.271 (26.18%)


,quantidade,percentual_do_total
motivo_descricao_ruim,,
unidade/embalagem,10670,16.17
marcador genérico,3070,4.65
dosagem/concentração,2962,4.49
registro/código,323,0.49
formatação inválida,244,0.37
expressão numérica,2,0.00


,motivo_descricao_ruim,marca_normalizada,quantidade
1122,dosagem/concentração,50 MG PO LIOF SOL IN,35
256,dosagem/concentração,100 U/ML SOL INJ CT,34
531,dosagem/concentração,1L,29
665,dosagem/concentração,200 MG COM REV CT BL,23
1244,dosagem/concentração,500ML,23
...,...,...,...
4863,unidade/embalagem,UND ALCOOL ETILICO,1
4868,unidade/embalagem,UNIDADE 500 ML,1
4871,unidade/embalagem,UNIDADE LITRO,1
4872,unidade/embalagem,UNIDADES - ANVISA -,1


### Exemplos contextualizados

Além da frequência dos valores, estes exemplos ajudam a conferir a descrição do item e a unidade de fornecimento associadas.

In [121]:
colunas_exemplo = [
    "motivo_descricao_ruim", "marca", "descricao_detalhada_item",
    "nome_unidade_fornecimento", "numero_controle_pncp",
]

if not descricoes_ruins.empty:
    # Remove repetições antes de selecionar exemplos, aumentando a diversidade da amostra.
    exemplos_descricoes = (
        descricoes_ruins.drop_duplicates(["motivo_descricao_ruim", "marca_normalizada"])
        .groupby("motivo_descricao_ruim", group_keys=False)
        .head(200)[colunas_exemplo]
    )
    display(exemplos_descricoes)

,motivo_descricao_ruim,marca,descricao_detalhada_item,nome_unidade_fornecimento,numero_controle_pncp
7,marcador genérico,Conforme TR,"Álcool Etílico Limpeza De Ambientes tipo: etílico, aplicação: limpeza, características adicionai...",LITRO,00000368000150-1-000110/2025
8,marcador genérico,DIVERSOS,"DIPIRONA SÓDICA, APRESENTAÇÃO ASSOCIADA À ESCOPOLAMINA BUTILBROMETO,COMPOSIÇÃO HOMATROPINA BUTIL...",COMPRIMIDO,00001727000193-1-000041/2024
10,unidade/embalagem,FRASCO 20ML,"CIANOCOBALAMINA, CONCENTRAÇÃO 7,5 MG/ML, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",FRASCO,00038174000143-1-000036/2023
24,formatação inválida,...,"Sal aplicação: alimentícia, características adicionais: teor mínimo cloreto de sódio 98,5%, tipo...",UNIDADE,00059311000126-1-000704/2025
28,marcador genérico,PRÓPRIA,"ÁCIDO CÍTRICO, ASPECTO FÍSICO PÓ, FÓRMULA QUÍMICA C6H8O7, PESO MOLECULAR192,12 G/MOL, CARACTERÍS...",QUILOGRAMA,00082024000137-1-000033/2024
...,...,...,...,...,...
57201,formatação inválida,"""ALCOOL; Tipo: Etíli","ÁLCOOL ETÍLICO, TIPO HIDRATADO, TEOR ALCOÓLICO 70%_(70°GL), APRESENTAÇÃOLÍQUIDO",FRASCO,56024581000156-1-000812/2024
57202,formatação inválida,"""ALCOOL; Tipo: Gel e","ÁLCOOL ETÍLICO, TIPO HIDRATADO, TEOR ALCOÓLICO 70%_(70°GL), APRESENTAÇÃO GEL",GALÃO,56024581000156-1-000812/2024
64040,formatação inválida,"""Clorexidina di","CLOREXIDINA DIGLUCONATO, DOSAGEM 0,5%, APLICAÇÃO SOLUÇÃO ALCOÓLICA",FRASCO,84012012000126-1-000098/2023
64387,formatação inválida,"""BICARBONATO DE SODI","BICARBONATO DE SÓDIO, CONCENTRAÇÃO 8,40%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL, CARACTERÍSTICA A...",AMPOLA,87252045000131-1-000267/2023


### Comparativo: lógica original × segunda rodada de melhoria

O comparativo abaixo preserva a lógica original em `possivelmente_validas_logica_original` e reúne todas as alterações posteriores em uma única **segunda rodada de melhoria**.

O quadro separa as linhas removidas pelas novas regras, os genéricos recuperados por decisão metodológica e a redução líquida. Assim, o resultado original de 75,94% permanece como referência permanente.

In [122]:
# Reconstitui os dois marcos oficiais e mantém os conjuntos disponíveis para inspeção.
possivelmente_valida_antes = ~(
    nula | em_branco | somente_numerica | descricao_nao_marca_antes
)
possivelmente_valida_sem_codelist = ~(
    nula | em_branco | somente_numerica | descricao_nao_marca_sem_codelist
)
possivelmente_valida_depois = ~(
    nula | em_branco | somente_numerica | descricao_nao_marca
)

possivelmente_validas_logica_original = dados.loc[possivelmente_valida_antes].copy()
reclassificadas_segunda_rodada = dados.loc[
    possivelmente_valida_antes & ~possivelmente_valida_depois
].copy()
reducao_liquida = int(possivelmente_valida_antes.sum() - possivelmente_valida_depois.sum())

comparativo_regras = pd.DataFrame(
    {
        "quantidade": [
            int(possivelmente_valida_antes.sum()),
            int(possivelmente_valida_depois.sum()),
            reducao_liquida,
        ],
    },
    index=[
        "possivelmente válidas — lógica original",
        "possivelmente válidas — segunda rodada",
        "redução líquida de possíveis marcas",
    ],
)
comparativo_regras["percentual_do_total"] = (
    comparativo_regras["quantidade"] / len(dados) * 100 if len(dados) else 0
).round(2)
display(comparativo_regras)

# Mede separadamente o efeito da codelist, sem tratá-la como uma nova rodada lógica.
comparativo_codelist = pd.DataFrame(
    {
        "quantidade": [
            int(possivelmente_valida_sem_codelist.sum()),
            int(possivelmente_valida_depois.sum()),
        ],
    },
    index=[
        "segunda rodada — antes da codelist",
        "segunda rodada — após a codelist",
    ],
)


,quantidade,percentual_do_total
possivelmente válidas — lógica original,50063,75.88
possivelmente válidas — segunda rodada,47859,72.54
redução líquida de possíveis marcas,2204,3.34


## 5. Existem linhas nulas ou em branco?

- **Nula:** ausência de valor no CSV, interpretada pelo pandas como `NaN`.
- **Em branco:** texto existente, mas vazio ou composto apenas por espaços.

Como não existe conteúdo de marca para exibir, a tabela de exemplos identifica as linhas por número de controle e mostra o contexto do item.

In [123]:
# A união é usada somente no total e na seleção de exemplos; o resumo preserva a distinção.
sem_conteudo = nula | em_branco
resumo_ausencias = pd.DataFrame(
    {
        "quantidade": [int(nula.sum()), int(em_branco.sum()), int(sem_conteudo.sum())],
    },
    index=["nula", "em branco", "total sem conteúdo"],
)
resumo_ausencias["percentual_do_total"] = (
    resumo_ausencias["quantidade"] / len(dados) * 100 if len(dados) else 0
).round(2)
display(resumo_ausencias)

if sem_conteudo.any():
    exemplos_ausencias = dados.loc[
        sem_conteudo,
        ["marca", "descricao_detalhada_item", "nome_unidade_fornecimento", "numero_controle_pncp"],
    ].copy()
    exemplos_ausencias.insert(0, "tipo_ausencia", "em branco")
    exemplos_ausencias.loc[nula[sem_conteudo], "tipo_ausencia"] = "nula"
    display(exemplos_ausencias)
else:
    print("Não há linhas nulas ou em branco na coluna marca.")

,quantidade,percentual_do_total
nula,48,0.07
em branco,0,0.00
total sem conteúdo,48,0.07


,tipo_ausencia,marca,descricao_detalhada_item,nome_unidade_fornecimento,numero_controle_pncp
241,nula,NaN,"Reagente adicional: com certificado de análise, apresentação 5: solução aquosa, componentes 5: á...",UNIDADE,00348003002245-1-000013/2025
4898,nula,NaN,"Tofacitinibe composição*: sal citrato, concentração: 5",COMPRIMIDO,00394502000144-1-000294/2025
4927,nula,NaN,"Insulina tipo: degludeca, concentração: 100, forma farmaceutica: solução injetável, caracteristi...",TUBETE,00394502000144-1-000795/2025
4934,nula,NaN,"Degarelix composição: sal acetato, concentração: 80, forma farmacêutica: pó liófilo p/ injetável...",FRASCO-AMPOLA,00394502000144-1-001182/2025
5000,nula,NaN,"Álcool Etílico tipo: hidratado, teor alcoólico: 70% ( 70°gl), apresentação: glicerinado, líquido",LITRO,00394502000144-1-002635/2025
5028,nula,NaN,"Paliperidona composição: na forma palmitato, concentração: 100, forma farmacêutica: suspensão in...",SERINGA,00394502000144-1-003514/2025
5029,nula,NaN,"Paliperidona composição: na forma palmitato, concentração: 100, forma farmacêutica: suspensão in...",SERINGA,00394502000144-1-003514/2025
5030,nula,NaN,"Paliperidona composição: na forma palmitato, concentração: 100, forma farmacêutica: suspensão in...",SERINGA,00394502000144-1-003514/2025
9885,nula,NaN,"Lidocaína Cloridrato composição: associada com norepinefrina, concentração: 3% + 1:50.000, forma...",TUBETE,00508903000188-1-003059/2025
9887,nula,NaN,"Fluoreto De Sódio concentração: 2%, forma farmacêutica: gel tixotrópico, característica adiciona...",FRASCO,00508903000188-1-003059/2025


## 6. Resultado consolidado

Para evitar dupla contagem, a classificação final usa categorias mutuamente exclusivas. Uma marca é considerada **válida ou possivelmente válida** quando está preenchida e não é somente numérica nem reconhecida pelas regras de descrição não-marca.

In [124]:
# Começa pela hipótese residual de possível validade e sobrescreve as linhas problemáticas.
# A ordem abaixo define a precedência e garante uma única categoria final por linha.
dados["classificacao_final"] = "válida ou possivelmente válida"
dados.loc[descricao_nao_marca, "classificacao_final"] = "descrição não-marca"
dados.loc[somente_numerica, "classificacao_final"] = "somente numérica"
dados.loc[em_branco, "classificacao_final"] = "em branco"
dados.loc[nula, "classificacao_final"] = "nula"

ordem = [
    "somente numérica", "descrição não-marca", "em branco", "nula",
    "válida ou possivelmente válida",
]
# Como as categorias finais são mutuamente exclusivas, seus percentuais somam 100%.
resumo_final = (
    dados["classificacao_final"].value_counts()
    .reindex(ordem, fill_value=0).rename("quantidade").to_frame()
)
resumo_final["percentual"] = (
    resumo_final["quantidade"] / len(dados) * 100 if len(dados) else 0
).round(2)
resumo_final.loc["total"] = [len(dados), 100.0 if len(dados) else 0.0]
display(resumo_final)

qtd_possivelmente_validas = resumo_final.loc["válida ou possivelmente válida", "quantidade"]
pct_possivelmente_validas = resumo_final.loc["válida ou possivelmente válida", "percentual"]
print(
    f"Estimativa final: {qtd_possivelmente_validas:,.0f} de {len(dados):,} linhas "
    f"({pct_possivelmente_validas:.2f}%) contêm marcas POSSIVELMENTE VÁLIDAS."
    .replace(",", ".")
)

,quantidade,percentual
classificacao_final,,
somente numérica,798.0,1.21
descrição não-marca,17271.0,26.18
em branco,0.0,0.00
nula,48.0,0.07
válida ou possivelmente válida,47859.0,72.54
total,65976.0,100.00


Estimativa final: 47.859 de 65.976 linhas (72.54%) contêm marcas POSSIVELMENTE VÁLIDAS.


## 7. Amostra das marcas possivelmente válidas

A amostra e os valores mais frequentes permitem uma última inspeção visual e ajudam a identificar novas regras para iterações futuras.

In [125]:
# Este grupo é candidato a análise posterior; as regras não confirmam a existência da marca.
possivelmente_validas = dados[
    dados["classificacao_final"].eq("válida ou possivelmente válida")
]

frequencia_validas = (
    possivelmente_validas.groupby("marca", dropna=False)
    .size().rename("quantidade").sort_values(ascending=False).to_frame()
)
display(frequencia_validas)
display(
    possivelmente_validas[
        ["marca", "descricao_detalhada_item", "nome_unidade_fornecimento", "numero_controle_pncp"]
    ]
)

,quantidade
marca,
GENERICO,1452
GENÉRICO,757
HIPOLABOR,666
EMS,648
TEUTO,537
...,...
Álcool Etílico Gel V,1
Álcool Etílico Gel P,1
Álcool Etílico Gel 7,1


,marca,descricao_detalhada_item,nome_unidade_fornecimento,numero_controle_pncp
0,DFL,"MEPIVACAÍNA CLORIDRATO, APRESENTAÇÃO ASSOCIADA COM EPINEFRINA, DOSAGEM 2% +1:100.000",TUBETE,00000368000150-1-000039/2025
1,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TUBETE,00000368000150-1-000039/2025
2,DFL,"BENZOCAÍNA, CONCENTRAÇÃO 20%, USO GEL TÓPICO",POTE,00000368000150-1-000039/2025
3,IODONTOSUL,"BICARBONATO DE SÓDIO, APRESENTAÇÃO PÓ",SACHÊ,00000368000150-1-000039/2025
4,DFL,"MEPIVACAÍNA CLORIDRATO, CONCENTRAÇÃO 3%, FORMA FARMACÊUTICA SOLUÇÃO INJETÁVEL",TUBETE,00000368000150-1-000046/2024
...,...,...,...,...
65970,EMS,Rivaroxabana concentração: 20,COMPRIMIDO,96291141000180-1-008869/2024
65971,AUROBINDO,Finasterida concentração: 5,CÁPSULA,96291141000180-1-008869/2024
65972,MEDLEY,"Travoprosta dosagem: 0,04, apresentação: solução oftálmica",FRASCO,96291141000180-1-008869/2024
65973,Clarity Alcool liq,"ÁLCOOL ETÍLICO, TIPO HIDRATADO, TEOR ALCOÓLICO 70%_(70°GL), APRESENTAÇÃOLÍQUIDO",FRASCO,96480850000103-1-000011/2025
